# Phase 3: Alpha Research & Statistical Learning  
## Notebook 03_07 — LSTM Forecaster Without Overclaiming Predictability

### Research question

How can an LSTM-style sequence forecaster be built for financial returns while avoiding exaggerated claims about predictability?

This notebook follows naturally from:

```text
03_06_arima_vs_lstm_benchmark
```

The previous notebook compared ARIMA-style forecasting with an LSTM benchmark. This notebook goes deeper into the **LSTM forecaster workflow**, with emphasis on research hygiene.

It deliberately avoids the claim that an LSTM can reliably predict markets.

Instead, it asks:

> If we build an LSTM return forecaster, how do we validate it honestly, compare it against baselines, diagnose overfitting, quantify uncertainty, and avoid confusing noisy statistical artefacts with tradable alpha?

It covers:

1. synthetic financial data with weak signal and high noise;
2. chronological train/validation/test split;
3. train-only scaling;
4. sequence-window construction;
5. baseline forecasts;
6. LSTM training if TensorFlow is available;
7. explicit fallback model if TensorFlow is unavailable;
8. validation-loss monitoring;
9. out-of-sample forecast metrics;
10. information coefficient and directional accuracy;
11. placebo target sanity check;
12. residual bootstrap uncertainty bands;
13. regime-dependent forecast diagnostics;
14. turnover and transaction-cost sensitivity;
15. model card and anti-overclaim checklist;
16. saved forecasts, metrics, and manifest.

The key idea:

> In financial forecasting, the default assumption should be that predictability is weak until proven otherwise out of sample, after costs, against dumb baselines.

## 1. Why this notebook exists

LSTMs are powerful sequence models.

But in finance, power is not enough.

Financial return forecasting has several brutal properties:

- low signal-to-noise ratio;
- non-stationarity;
- regime changes;
- heavy tails;
- transaction costs;
- data leakage risks;
- multiple-testing risks;
- unstable relationships.

So the goal is not to say:

> "The LSTM predicts the market."

The goal is to build a pipeline that can honestly say:

> "Here is what the LSTM forecasted, here is how it compared to baselines, here is where it failed, here is the uncertainty, and here is why we should be cautious."

## 2. Forecast target

We forecast next-period return:

$$
r_{t+1}
$$

not raw price.

The model receives a sequence of features:

$$
X_t = [x_{t-L+1},\dots,x_t]
$$

and produces:

$$
\hat r_{t+1}
$$

A return forecast can be converted into a price forecast:

$$
\hat P_{t+1}=P_t e^{\hat r_{t+1}}
$$

But evaluation should focus on returns, forecast errors, uncertainty, and economic usefulness after costs.

## 3. Validation philosophy

This notebook uses:

1. chronological train/validation/test split;
2. train-only scaling;
3. no random row shuffling across time;
4. baseline comparisons;
5. placebo tests;
6. regime diagnostics;
7. cost sensitivity;
8. conservative language.

A model that only beats baselines in-sample is not useful.

A model that beats baselines out-of-sample but fails after costs is also not useful.

A model that works only in one volatility regime needs to be reported as such.

## 4. Imports and configuration

In [ ]:
from __future__ import annotations

from dataclasses import asdict, dataclass
from pathlib import Path
import json
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    import tensorflow as tf
    from tensorflow import keras
    TENSORFLOW_AVAILABLE = True
except Exception:
    TENSORFLOW_AVAILABLE = False

try:
    from sklearn.neural_network import MLPRegressor
    SKLEARN_AVAILABLE = True
except Exception:
    SKLEARN_AVAILABLE = False

TENSORFLOW_AVAILABLE, SKLEARN_AVAILABLE

In [ ]:
@dataclass(frozen=True)
class LSTMForecasterConfig:
    n_obs: int = 3_500
    train_fraction: float = 0.60
    validation_fraction: float = 0.20
    sequence_length: int = 60
    annualisation: int = 252
    rolling_vol_window: int = 21
    long_momentum_window: int = 63
    lstm_units: int = 32
    dense_units: int = 16
    dropout_rate: float = 0.20
    epochs: int = 30
    batch_size: int = 64
    patience: int = 6
    seed: int = 42


config = LSTMForecasterConfig()
config

## 5. Synthetic data with weak predictability

We create synthetic data with:

1. weak autoregressive structure;
2. slow momentum component;
3. volatility clustering;
4. regime-dependent volatility;
5. heavy-tailed shocks;
6. nonlinear but weak signal.

This is not meant to make forecasting easy.

The true signal is deliberately weak so the notebook behaves like realistic financial forecasting: noisy and fragile.

In [ ]:
def simulate_weakly_predictable_market(config: LSTMForecasterConfig) -> pd.DataFrame:
    rng = np.random.default_rng(config.seed)
    n = config.n_obs

    returns = np.zeros(n)
    latent_signal = np.zeros(n)
    latent_vol = np.zeros(n)
    regime = np.zeros(n, dtype=int)

    transition = np.array([
        [0.985, 0.015],
        [0.040, 0.960]
    ])

    regime_vol = np.array([0.008, 0.022])

    latent_vol[0] = regime_vol[0]

    for t in range(3, n):
        regime[t] = rng.choice([0, 1], p=transition[regime[t - 1]])

        latent_signal[t] = (
            0.94 * latent_signal[t - 1]
            + 0.04 * np.tanh(returns[t - 1] / 0.015)
            - 0.02 * np.tanh(returns[t - 2] / 0.015)
            + 0.02 * rng.standard_normal()
        )

        latent_vol[t] = (
            0.93 * latent_vol[t - 1]
            + 0.07 * regime_vol[regime[t]]
            + 0.10 * abs(returns[t - 1])
        )
        latent_vol[t] = np.clip(latent_vol[t], 0.003, 0.070)

        weak_predictable_mean = 0.00008 + 0.00035 * latent_signal[t - 1]
        seasonal_noise = 0.00015 * np.sin(2 * np.pi * t / 126)

        shock = rng.standard_t(df=6) * np.sqrt((6 - 2) / 6)

        returns[t] = weak_predictable_mean + seasonal_noise + latent_vol[t] * shock

    dates = pd.bdate_range("2015-01-01", periods=n)
    price = 100 * np.exp(np.cumsum(returns))

    out = pd.DataFrame({
        "timestamp": dates,
        "return": returns,
        "price": price,
        "latent_signal": latent_signal,
        "latent_vol": latent_vol,
        "regime": regime
    })

    out["abs_return"] = out["return"].abs()
    out["squared_return"] = out["return"] ** 2
    out["rolling_vol_21"] = out["return"].rolling(config.rolling_vol_window).std()
    out["rolling_mean_5"] = out["return"].rolling(5).mean()
    out["rolling_mean_21"] = out["return"].rolling(21).mean()
    out["momentum_21"] = np.log(out["price"] / out["price"].shift(21))
    out["momentum_63"] = np.log(out["price"] / out["price"].shift(config.long_momentum_window))
    out["drawdown_63"] = out["price"] / out["price"].rolling(config.long_momentum_window).max() - 1.0
    out["vol_ratio_21_63"] = out["rolling_vol_21"] / out["return"].rolling(config.long_momentum_window).std()
    out["target_next_return"] = out["return"].shift(-1)
    out["target_next_price"] = out["price"].shift(-1)

    out = out.replace([np.inf, -np.inf], np.nan).dropna().reset_index(drop=True)

    return out


data = simulate_weakly_predictable_market(config)

data.head()

In [ ]:
plt.figure(figsize=(12, 6))
plt.plot(data["timestamp"], data["price"])
plt.title("Synthetic Price Series")
plt.xlabel("Date")
plt.ylabel("Price")
plt.show()

plt.figure(figsize=(12, 6))
plt.plot(data["timestamp"], data["return"])
plt.title("Synthetic Returns")
plt.xlabel("Date")
plt.ylabel("Return")
plt.show()

plt.figure(figsize=(12, 4))
plt.plot(data["timestamp"], data["latent_vol"] * np.sqrt(config.annualisation), label="Latent annualised volatility")
plt.title("Latent Volatility")
plt.xlabel("Date")
plt.ylabel("Annualised volatility")
plt.legend()
plt.show()

## 6. Basic data diagnostics

We inspect:

- distribution of returns;
- volatility clustering;
- target autocorrelation;
- baseline predictability.

If these diagnostics look weak, that is not a failure. That is normal in financial return prediction.

In [ ]:
def autocorrelation(x, max_lag):
    x = np.asarray(x, dtype=float)
    x = x - np.mean(x)
    denom = np.sum(x ** 2)

    rows = []
    for lag in range(1, max_lag + 1):
        rows.append({
            "lag": lag,
            "autocorrelation": float(np.sum(x[lag:] * x[:-lag]) / denom)
        })

    return pd.DataFrame(rows)


acf_returns = autocorrelation(data["return"], 40).assign(series="return")
acf_abs = autocorrelation(data["abs_return"], 40).assign(series="absolute_return")
acf_sq = autocorrelation(data["squared_return"], 40).assign(series="squared_return")
acf_table = pd.concat([acf_returns, acf_abs, acf_sq], ignore_index=True)

summary_stats = pd.Series({
    "daily_mean_return": data["return"].mean(),
    "daily_volatility": data["return"].std(),
    "annualised_mean_return": data["return"].mean() * config.annualisation,
    "annualised_volatility": data["return"].std() * np.sqrt(config.annualisation),
    "skew": data["return"].skew(),
    "excess_kurtosis": data["return"].kurtosis(),
    "target_autocorr_lag1": data["target_next_return"].autocorr(lag=1)
})

summary_stats

In [ ]:
plt.figure(figsize=(10, 6))
for name, group in acf_table.groupby("series"):
    plt.plot(group["lag"], group["autocorrelation"], marker="o", label=name)
plt.axhline(0, linestyle="--")
plt.title("Autocorrelation Diagnostics")
plt.xlabel("Lag")
plt.ylabel("Autocorrelation")
plt.legend()
plt.show()

plt.figure(figsize=(10, 6))
plt.hist(data["return"], bins=100, density=True)
plt.title("Return Distribution")
plt.xlabel("Return")
plt.ylabel("Density")
plt.show()

## 7. Chronological split

We split into:

1. train;
2. validation;
3. test.

The test set is untouched until final reporting.

In [ ]:
n = len(data)
train_end = int(config.train_fraction * n)
validation_end = int((config.train_fraction + config.validation_fraction) * n)

train_df = data.iloc[:train_end].copy()
val_df = data.iloc[train_end:validation_end].copy()
test_df = data.iloc[validation_end:].copy()

split_metadata = {
    "n_total": int(n),
    "n_train": int(len(train_df)),
    "n_validation": int(len(val_df)),
    "n_test": int(len(test_df)),
    "train_start": str(train_df["timestamp"].iloc[0]),
    "train_end": str(train_df["timestamp"].iloc[-1]),
    "validation_start": str(val_df["timestamp"].iloc[0]),
    "validation_end": str(val_df["timestamp"].iloc[-1]),
    "test_start": str(test_df["timestamp"].iloc[0]),
    "test_end": str(test_df["timestamp"].iloc[-1])
}

pd.Series(split_metadata)

## 8. Feature set

The LSTM receives a window of engineered features.

Features:

1. return;
2. absolute return;
3. rolling volatility;
4. short rolling mean;
5. medium rolling mean;
6. 21-day momentum;
7. 63-day momentum;
8. drawdown;
9. volatility ratio.

These are all computed using current and past information only.

In [ ]:
feature_cols = [
    "return",
    "abs_return",
    "rolling_vol_21",
    "rolling_mean_5",
    "rolling_mean_21",
    "momentum_21",
    "momentum_63",
    "drawdown_63",
    "vol_ratio_21_63"
]

target_col = "target_next_return"

feature_summary = train_df[feature_cols].describe().T

feature_summary

## 9. Train-only scaling

Scaling is a common source of leakage.

We fit the scaler only on the training set:

$$
z = \frac{x-\mu_{\text{train}}}{\sigma_{\text{train}}}
$$

The validation and test sets use the same train parameters.

In [ ]:
def fit_scaler(df, cols):
    mean = df[cols].mean()
    std = df[cols].std(ddof=1).replace(0, 1.0)
    return mean, std


feature_mean, feature_std = fit_scaler(train_df, feature_cols)
target_mean = train_df[target_col].mean()
target_std = train_df[target_col].std(ddof=1)


def apply_scaling(df, feature_cols, feature_mean, feature_std, target_mean, target_std):
    out = df.copy()

    for col in feature_cols:
        out[f"{col}_scaled"] = (out[col] - feature_mean[col]) / feature_std[col]

    out["target_scaled"] = (out[target_col] - target_mean) / target_std

    return out


scaled_data = apply_scaling(data, feature_cols, feature_mean, feature_std, target_mean, target_std)
scaled_feature_cols = [f"{c}_scaled" for c in feature_cols]

scaled_data[["timestamp"] + scaled_feature_cols + ["target_scaled"]].head()

## 10. Sequence-window construction

For each timestamp, the input is a trailing sequence of length $L$:

$$
X_t = [x_{t-L+1},\dots,x_t]
$$

The label is:

$$
r_{t+1}
$$

We keep timestamp metadata so forecasts can be aligned with validation/test sets.

In [ ]:
def make_sequence_dataset(df, feature_cols, target_col, sequence_length):
    values = df[feature_cols].to_numpy(dtype=float)
    targets = df[target_col].to_numpy(dtype=float)
    timestamps = df["timestamp"].to_numpy()

    X = []
    y = []
    ts = []

    for t in range(sequence_length - 1, len(df)):
        X.append(values[t-sequence_length+1:t+1])
        y.append(targets[t])
        ts.append(timestamps[t])

    return np.asarray(X), np.asarray(y), np.asarray(ts)


X_all, y_all, ts_all = make_sequence_dataset(
    scaled_data,
    scaled_feature_cols,
    "target_scaled",
    config.sequence_length
)

sequence_meta = pd.DataFrame({"timestamp": pd.to_datetime(ts_all)})

train_end_date = train_df["timestamp"].iloc[-1]
val_end_date = val_df["timestamp"].iloc[-1]

train_mask = sequence_meta["timestamp"] <= train_end_date
val_mask = (sequence_meta["timestamp"] > train_end_date) & (sequence_meta["timestamp"] <= val_end_date)
test_mask = sequence_meta["timestamp"] > val_end_date

X_train, y_train = X_all[train_mask], y_all[train_mask]
X_val, y_val = X_all[val_mask], y_all[val_mask]
X_test, y_test = X_all[test_mask], y_all[test_mask]
ts_test = sequence_meta.loc[test_mask, "timestamp"].to_numpy()

pd.Series({
    "X_train_shape": str(X_train.shape),
    "X_val_shape": str(X_val.shape),
    "X_test_shape": str(X_test.shape)
})

## 11. Baseline forecasts

We include baselines because an LSTM must beat something simple.

Baselines:

1. zero return;
2. train mean return;
3. last return;
4. rolling mean return;
5. momentum sign forecast.

These are intentionally simple.

In [ ]:
def make_baseline_forecasts(df, train_df):
    out = df[["timestamp", "price", "return", "target_next_return", "target_next_price", "regime", "latent_vol"]].copy()

    train_mean = train_df["target_next_return"].mean()

    out["forecast_zero"] = 0.0
    out["forecast_train_mean"] = train_mean
    out["forecast_last_return"] = out["return"]
    out["forecast_rolling_mean_20"] = out["return"].rolling(20).mean().fillna(train_mean)

    momentum_scale = train_df["target_next_return"].abs().mean()
    out["forecast_momentum_sign"] = np.sign(out["momentum_21"]) * 0.25 * momentum_scale

    return out


test_forecasts = make_baseline_forecasts(test_df, train_df)

baseline_cols = [
    "forecast_zero",
    "forecast_train_mean",
    "forecast_last_return",
    "forecast_rolling_mean_20",
    "forecast_momentum_sign"
]

test_forecasts.head()

## 12. Training the LSTM or fallback model

If TensorFlow is available, we train a compact LSTM.

If TensorFlow is unavailable, we train a clearly labelled fallback model:

```text
sklearn MLP on flattened sequence windows
```

That fallback is **not** an LSTM. It exists only so the notebook remains runnable.

In [ ]:
def train_lstm_or_fallback(config, X_train, y_train, X_val, y_val):
    if TENSORFLOW_AVAILABLE:
        tf.random.set_seed(config.seed)

        model = keras.Sequential([
            keras.layers.Input(shape=(X_train.shape[1], X_train.shape[2])),
            keras.layers.LSTM(config.lstm_units, return_sequences=False),
            keras.layers.Dropout(config.dropout_rate),
            keras.layers.Dense(config.dense_units, activation="relu"),
            keras.layers.Dropout(config.dropout_rate),
            keras.layers.Dense(1)
        ])

        model.compile(
            optimizer=keras.optimizers.Adam(learning_rate=0.001),
            loss="mse"
        )

        callbacks = [
            keras.callbacks.EarlyStopping(
                monitor="val_loss",
                patience=config.patience,
                restore_best_weights=True
            )
        ]

        history = model.fit(
            X_train,
            y_train,
            validation_data=(X_val, y_val),
            epochs=config.epochs,
            batch_size=config.batch_size,
            verbose=0,
            callbacks=callbacks
        )

        return {
            "model": model,
            "model_type": "tensorflow_lstm",
            "history": pd.DataFrame(history.history),
            "available": True
        }

    if SKLEARN_AVAILABLE:
        X_train_flat = X_train.reshape(X_train.shape[0], -1)

        model = MLPRegressor(
            hidden_layer_sizes=(64, 32),
            activation="relu",
            alpha=1e-4,
            learning_rate_init=1e-3,
            max_iter=300,
            early_stopping=True,
            validation_fraction=0.15,
            random_state=config.seed
        )

        model.fit(X_train_flat, y_train)

        history = pd.DataFrame({
            "loss": getattr(model, "loss_curve_", [])
        })

        return {
            "model": model,
            "model_type": "sklearn_mlp_flattened_sequence_fallback_not_lstm",
            "history": history,
            "available": True
        }

    return {
        "model": None,
        "model_type": "no_neural_model_available",
        "history": pd.DataFrame(),
        "available": False
    }


model_fit = train_lstm_or_fallback(config, X_train, y_train, X_val, y_val)

model_fit["model_type"], model_fit["available"]

In [ ]:
if not model_fit["history"].empty:
    plt.figure(figsize=(10, 6))
    for col in model_fit["history"].columns:
        plt.plot(model_fit["history"][col], label=col)
    plt.title(f"Training History: {model_fit['model_type']}")
    plt.xlabel("Epoch / iteration")
    plt.ylabel("Loss")
    plt.legend()
    plt.show()
else:
    print("No training history available.")

## 13. Test forecasts

Predictions are made in scaled target space and then inverse-transformed:

$$
\hat r = \hat z\sigma_{\text{train}}+\mu_{\text{train}}
$$

In [ ]:
def predict_model(model_fit, X):
    if not model_fit["available"]:
        return np.full(X.shape[0], np.nan)

    if model_fit["model_type"] == "tensorflow_lstm":
        pred_scaled = model_fit["model"].predict(X, verbose=0).reshape(-1)
    else:
        X_flat = X.reshape(X.shape[0], -1)
        pred_scaled = model_fit["model"].predict(X_flat).reshape(-1)

    return pred_scaled * target_std + target_mean


lstm_predictions = predict_model(model_fit, X_test)

lstm_forecasts = pd.DataFrame({
    "timestamp": pd.to_datetime(ts_test),
    "forecast_lstm": lstm_predictions
})

test_forecasts = test_forecasts.merge(lstm_forecasts, on="timestamp", how="left")

test_forecasts[["timestamp", "forecast_lstm", "target_next_return"]].dropna().head()

## 14. Forecast metrics

We evaluate:

1. MSE;
2. MAE;
3. forecast/actual correlation;
4. information coefficient;
5. directional accuracy;
6. sign-strategy information ratio.

The information coefficient is simply the correlation between forecast and future return.

In [ ]:
def forecast_metrics(df, forecast_col):
    subset = df.dropna(subset=[forecast_col, "target_next_return"]).copy()

    y = subset["target_next_return"].to_numpy()
    pred = subset[forecast_col].to_numpy()

    error = pred - y
    sign_strategy = np.sign(pred) * y

    corr = np.corrcoef(pred, y)[0, 1] if np.std(pred) > 0 and np.std(y) > 0 else np.nan

    return {
        "forecast": forecast_col,
        "n": int(len(subset)),
        "mse": float(np.mean(error ** 2)),
        "mae": float(np.mean(np.abs(error))),
        "information_coefficient": float(corr),
        "directional_accuracy": float(np.mean(np.sign(pred) == np.sign(y))),
        "mean_forecast": float(np.mean(pred)),
        "std_forecast": float(np.std(pred, ddof=1)),
        "mean_target": float(np.mean(y)),
        "std_target": float(np.std(y, ddof=1)),
        "sign_strategy_ann_return": float(np.mean(sign_strategy) * config.annualisation),
        "sign_strategy_ann_vol": float(np.std(sign_strategy, ddof=1) * np.sqrt(config.annualisation)),
        "sign_strategy_ir": float(np.mean(sign_strategy) / np.std(sign_strategy, ddof=1) * np.sqrt(config.annualisation)) if np.std(sign_strategy, ddof=1) > 0 else np.nan
    }


forecast_cols = baseline_cols + ["forecast_lstm"]

metrics = pd.DataFrame([
    forecast_metrics(test_forecasts, col)
    for col in forecast_cols
]).sort_values("mse")

metrics

In [ ]:
plt.figure(figsize=(12, 6))
plt.barh(metrics["forecast"], metrics["mse"])
plt.title("Forecast MSE, Lower is Better")
plt.xlabel("MSE")
plt.ylabel("Forecast")
plt.show()

plt.figure(figsize=(12, 6))
plot_df = metrics.sort_values("information_coefficient")
plt.barh(plot_df["forecast"], plot_df["information_coefficient"])
plt.axvline(0, linestyle="--")
plt.title("Information Coefficient")
plt.xlabel("Forecast/target correlation")
plt.ylabel("Forecast")
plt.show()

## 15. Visual forecast diagnostics

Return forecasts should be small and noisy.

If a model produces huge confident forecasts, that is usually a red flag.

We plot:

1. actual next returns;
2. LSTM forecasts;
3. baseline forecasts.

In [ ]:
aligned = test_forecasts.dropna(subset=["forecast_lstm", "target_next_return"]).copy()
plot_sample = aligned.iloc[:300]

plt.figure(figsize=(12, 6))
plt.plot(plot_sample["timestamp"], plot_sample["target_next_return"], label="Actual next return", alpha=0.65)
plt.plot(plot_sample["timestamp"], plot_sample["forecast_lstm"], label="LSTM forecast", alpha=0.85)
plt.plot(plot_sample["timestamp"], plot_sample["forecast_zero"], label="Zero forecast", alpha=0.75)
plt.axhline(0, linestyle="--")
plt.title("Forecasts vs Realised Returns, Test Sample")
plt.xlabel("Date")
plt.ylabel("Return")
plt.legend()
plt.show()

plt.figure(figsize=(10, 6))
plt.scatter(aligned["forecast_lstm"], aligned["target_next_return"], alpha=0.35)
plt.axhline(0, linestyle="--")
plt.axvline(0, linestyle="--")
plt.title("LSTM Forecast vs Realised Next Return")
plt.xlabel("Forecast return")
plt.ylabel("Realised next return")
plt.show()

## 16. Residual diagnostics

Forecast error:

$$
e_{t+1}=r_{t+1}-\hat r_{t+1}
$$

We inspect:

- mean error;
- error volatility;
- error autocorrelation;
- error by forecast magnitude.

If the residuals still contain strong structure, the model is missing something.

If forecast magnitude does not correspond to realised outcome, the model may not be useful for sizing.

In [ ]:
aligned["lstm_error"] = aligned["target_next_return"] - aligned["forecast_lstm"]
aligned["abs_lstm_error"] = aligned["lstm_error"].abs()
aligned["forecast_abs_bucket"] = pd.qcut(
    aligned["forecast_lstm"].abs(),
    q=4,
    labels=["small", "medium", "large", "very_large"],
    duplicates="drop"
)

residual_summary = pd.Series({
    "mean_error": aligned["lstm_error"].mean(),
    "median_error": aligned["lstm_error"].median(),
    "error_std": aligned["lstm_error"].std(),
    "mean_abs_error": aligned["abs_lstm_error"].mean(),
    "error_autocorr_lag1": aligned["lstm_error"].autocorr(lag=1)
})

bucket_error = (
    aligned
    .groupby("forecast_abs_bucket", observed=True)
    .agg(
        n=("lstm_error", "count"),
        mean_abs_forecast=("forecast_lstm", lambda x: np.mean(np.abs(x))),
        mean_abs_error=("abs_lstm_error", "mean"),
        directional_accuracy=("target_next_return", lambda x: np.nan)
    )
    .reset_index()
)

direction_by_bucket = []
for bucket, group in aligned.groupby("forecast_abs_bucket", observed=True):
    direction_by_bucket.append((bucket, np.mean(np.sign(group["forecast_lstm"]) == np.sign(group["target_next_return"]))))

direction_map = dict(direction_by_bucket)
bucket_error["directional_accuracy"] = bucket_error["forecast_abs_bucket"].map(direction_map).astype(float)

residual_summary

In [ ]:
bucket_error

In [ ]:
plt.figure(figsize=(10, 6))
plt.hist(aligned["lstm_error"], bins=80)
plt.axvline(0, linestyle="--")
plt.title("LSTM Forecast Error Distribution")
plt.xlabel("Error: actual - forecast")
plt.ylabel("Count")
plt.show()

plt.figure(figsize=(10, 6))
plt.bar(bucket_error["forecast_abs_bucket"].astype(str), bucket_error["directional_accuracy"])
plt.axhline(0.5, linestyle="--")
plt.title("Directional Accuracy by Forecast Magnitude Bucket")
plt.xlabel("Absolute forecast bucket")
plt.ylabel("Directional accuracy")
plt.show()

## 17. Placebo target sanity check

A common failure mode:

> A model appears to predict returns because the validation protocol is flawed.

We run a placebo test by shuffling the training labels while preserving feature sequences.

A valid pipeline should not produce meaningful out-of-sample performance on shuffled labels.

If placebo performance is similar to real-label performance, the signal is suspicious.

In [ ]:
def train_placebo_model(config, X_train, y_train, X_val, y_val):
    rng = np.random.default_rng(config.seed + 999)
    y_train_shuffled = y_train.copy()
    rng.shuffle(y_train_shuffled)

    # Keep validation labels real for monitoring; the model should fail to generalise.
    return train_lstm_or_fallback(config, X_train, y_train_shuffled, X_val, y_val)


placebo_fit = train_placebo_model(config, X_train, y_train, X_val, y_val)
placebo_predictions = predict_model(placebo_fit, X_test)

placebo_forecasts = pd.DataFrame({
    "timestamp": pd.to_datetime(ts_test),
    "forecast_placebo": placebo_predictions
})

test_forecasts = test_forecasts.merge(placebo_forecasts, on="timestamp", how="left")

placebo_metrics = pd.DataFrame([
    forecast_metrics(test_forecasts, "forecast_lstm"),
    forecast_metrics(test_forecasts, "forecast_placebo"),
    forecast_metrics(test_forecasts, "forecast_zero")
])

placebo_metrics

## 18. Residual bootstrap uncertainty bands

Point forecasts alone are misleading.

We estimate forecast uncertainty by resampling residuals from the validation set.

Procedure:

1. predict validation set;
2. compute validation residuals;
3. for each test forecast, add bootstrapped residuals;
4. estimate forecast interval quantiles.

This is a simple residual bootstrap, not a full probabilistic model.

In [ ]:
# Validation predictions for residual bootstrap.
val_predictions = predict_model(model_fit, X_val)
val_actual = y_val * target_std + target_mean
val_residuals = val_actual - val_predictions

rng = np.random.default_rng(config.seed + 1234)
n_boot = 1000

test_pred = aligned["forecast_lstm"].to_numpy()
boot_residuals = rng.choice(val_residuals, size=(len(test_pred), n_boot), replace=True)
boot_scenarios = test_pred[:, None] + boot_residuals

lower_05 = np.quantile(boot_scenarios, 0.05, axis=1)
upper_95 = np.quantile(boot_scenarios, 0.95, axis=1)

uncertainty_df = aligned[["timestamp", "forecast_lstm", "target_next_return"]].copy()
uncertainty_df["lower_05"] = lower_05
uncertainty_df["upper_95"] = upper_95
uncertainty_df["covered_90_interval"] = (
    (uncertainty_df["target_next_return"] >= uncertainty_df["lower_05"])
    & (uncertainty_df["target_next_return"] <= uncertainty_df["upper_95"])
)

uncertainty_summary = pd.Series({
    "nominal_interval": 0.90,
    "empirical_coverage": uncertainty_df["covered_90_interval"].mean(),
    "mean_interval_width": (uncertainty_df["upper_95"] - uncertainty_df["lower_05"]).mean(),
    "validation_residual_std": np.std(val_residuals, ddof=1)
})

uncertainty_summary

In [ ]:
plot_unc = uncertainty_df.iloc[:250]

plt.figure(figsize=(12, 6))
plt.plot(plot_unc["timestamp"], plot_unc["target_next_return"], label="Actual", alpha=0.65)
plt.plot(plot_unc["timestamp"], plot_unc["forecast_lstm"], label="Forecast", alpha=0.9)
plt.fill_between(
    plot_unc["timestamp"],
    plot_unc["lower_05"],
    plot_unc["upper_95"],
    alpha=0.25,
    label="Residual bootstrap 90% interval"
)
plt.axhline(0, linestyle="--")
plt.title("LSTM Forecast with Residual Bootstrap Interval")
plt.xlabel("Date")
plt.ylabel("Return")
plt.legend()
plt.show()

## 19. Regime diagnostics

A model can look acceptable on average but fail in stress regimes.

We bucket test observations by realised absolute return and latent volatility regime.

In real data, latent regime is unavailable, so realised-volatility buckets are more realistic.

In [ ]:
aligned["realized_abs_return"] = aligned["target_next_return"].abs()
aligned["realized_vol_bucket"] = pd.qcut(
    aligned["realized_abs_return"],
    q=4,
    labels=["low", "mid_low", "mid_high", "high"],
    duplicates="drop"
)

regime_rows = []

for bucket, group in aligned.groupby("realized_vol_bucket", observed=True):
    for col in ["forecast_lstm", "forecast_zero", "forecast_rolling_mean_20"]:
        err = group[col] - group["target_next_return"]
        regime_rows.append({
            "bucket": str(bucket),
            "forecast": col,
            "n": len(group),
            "mse": float(np.mean(err ** 2)),
            "mae": float(np.mean(np.abs(err))),
            "directional_accuracy": float(np.mean(np.sign(group[col]) == np.sign(group["target_next_return"]))),
            "information_coefficient": float(np.corrcoef(group[col], group["target_next_return"])[0, 1]) if np.std(group[col]) > 0 else np.nan
        })

regime_diagnostics = pd.DataFrame(regime_rows)

regime_diagnostics

In [ ]:
plt.figure(figsize=(10, 6))
for forecast, group in regime_diagnostics.groupby("forecast"):
    plt.plot(group["bucket"], group["mse"], marker="o", label=forecast)
plt.title("Forecast MSE by Realised Volatility Bucket")
plt.xlabel("Realised absolute return bucket")
plt.ylabel("MSE")
plt.legend()
plt.show()

## 20. Toy strategy and transaction costs

A forecast is not alpha unless it survives costs.

We use a toy strategy:

$$
w_t=\operatorname{clip}\Big(\frac{\hat r_{t+1}}{2\sigma_{\hat r}},-1,1\Big)
$$

Strategy return:

$$
R_{t+1}=w_t r_{t+1}-c|w_t-w_{t-1}|
$$

We test several cost levels.

This is a diagnostic only, not a trading recommendation.

In [ ]:
def toy_strategy(df, forecast_col, cost):
    out = df[["timestamp", forecast_col, "target_next_return"]].dropna().copy()

    forecast_std = out[forecast_col].std()
    forecast_std = forecast_std if forecast_std > 1e-12 else 1e-4

    out["weight"] = np.clip(out[forecast_col] / (2.0 * forecast_std), -1.0, 1.0)
    out["turnover"] = out["weight"].diff().abs().fillna(out["weight"].abs())
    out["strategy_return"] = out["weight"] * out["target_next_return"] - cost * out["turnover"]
    out["cum_growth"] = np.exp(out["strategy_return"].cumsum())

    return out


cost_levels = [0.0, 0.00002, 0.00005, 0.00010, 0.00020]
strategy_frames = []
strategy_summary_rows = []

for cost in cost_levels:
    for col in ["forecast_lstm", "forecast_zero", "forecast_rolling_mean_20", "forecast_momentum_sign"]:
        strat = toy_strategy(test_forecasts, col, cost=cost)
        strat["forecast"] = col
        strat["cost"] = cost
        strategy_frames.append(strat)

        ret = strat["strategy_return"]
        strategy_summary_rows.append({
            "forecast": col,
            "cost": cost,
            "annualised_return": ret.mean() * config.annualisation,
            "annualised_vol": ret.std(ddof=1) * np.sqrt(config.annualisation),
            "information_ratio": ret.mean() / ret.std(ddof=1) * np.sqrt(config.annualisation) if ret.std(ddof=1) > 0 else np.nan,
            "average_abs_weight": strat["weight"].abs().mean(),
            "average_turnover": strat["turnover"].mean(),
            "total_log_return": ret.sum()
        })

strategy_df = pd.concat(strategy_frames, ignore_index=True)
strategy_summary = pd.DataFrame(strategy_summary_rows)

strategy_summary.sort_values(["cost", "information_ratio"], ascending=[True, False]).head(10)

In [ ]:
plt.figure(figsize=(10, 6))
for forecast, group in strategy_summary.groupby("forecast"):
    plt.plot(group["cost"], group["information_ratio"], marker="o", label=forecast)
plt.axhline(0, linestyle="--")
plt.title("Toy Strategy Information Ratio vs Transaction Cost")
plt.xlabel("Cost per unit turnover")
plt.ylabel("Information ratio")
plt.legend()
plt.show()

## 21. Forecast model card

A responsible model report should include:

| Item | Status |
|---|---|
| Forecast target | next-period return |
| Split type | chronological |
| Scaling | train-only |
| Baselines | included |
| Placebo test | included |
| Costs | sensitivity tested |
| Regime diagnostics | included |
| Claim of tradable alpha | not made |

This notebook should be read as a research infrastructure notebook, not as a claim that LSTMs can predict markets.

In [ ]:
model_card = pd.Series({
    "model_name": "03_07_lstm_forecaster",
    "model_type": model_fit["model_type"],
    "forecast_target": "next_period_return",
    "uses_chronological_split": True,
    "uses_train_only_scaling": True,
    "baselines_included": True,
    "placebo_test_included": True,
    "uncertainty_bands_included": True,
    "transaction_cost_sensitivity_included": True,
    "overclaiming_warning": "No claim of reliable market predictability or tradable alpha.",
    "tensorflow_available": TENSORFLOW_AVAILABLE,
    "sklearn_available": SKLEARN_AVAILABLE
})

model_card

## 22. Anti-overclaim checklist

Before presenting an LSTM forecast as useful, ask:

1. Does it beat zero and naïve baselines out of sample?
2. Does it beat them on aligned timestamps?
3. Does it survive transaction costs?
4. Is the information coefficient stable?
5. Does it work outside one lucky regime?
6. Does it pass placebo tests?
7. Are scalers fit on training data only?
8. Is the test set untouched until final reporting?
9. Are uncertainty bands reported?
10. Are forecasts economically large enough to trade?
11. Are turnover and capacity considered?
12. Is there a plausible economic mechanism?

If the answer is no, the model may still be a useful experiment, but not a credible alpha claim.

## 23. Saving outputs

The notebook saves:

1. synthetic market data;
2. feature summary;
3. split metadata;
4. training history;
5. test forecasts;
6. forecast metrics;
7. residual diagnostics;
8. placebo metrics;
9. uncertainty intervals;
10. regime diagnostics;
11. transaction-cost strategy results;
12. model card;
13. manifest.

In [ ]:
output_dir = Path("data/processed/lstm_forecaster_v1")
output_dir.mkdir(parents=True, exist_ok=True)

data_path = output_dir / "synthetic_market_data.csv"
feature_summary_path = output_dir / "feature_summary.csv"
summary_stats_path = output_dir / "summary_statistics.csv"
acf_path = output_dir / "autocorrelation_diagnostics.csv"
split_path = output_dir / "split_metadata.json"
training_history_path = output_dir / "training_history.csv"
test_forecasts_path = output_dir / "test_forecasts.csv"
metrics_path = output_dir / "forecast_metrics.csv"
residual_summary_path = output_dir / "residual_summary.csv"
bucket_error_path = output_dir / "forecast_magnitude_bucket_error.csv"
placebo_metrics_path = output_dir / "placebo_metrics.csv"
uncertainty_path = output_dir / "forecast_uncertainty_intervals.csv"
uncertainty_summary_path = output_dir / "uncertainty_summary.csv"
regime_path = output_dir / "regime_diagnostics.csv"
strategy_path = output_dir / "toy_strategy_results.csv"
strategy_summary_path = output_dir / "toy_strategy_summary.csv"
model_card_path = output_dir / "model_card.csv"
manifest_path = output_dir / "manifest.json"

data.to_csv(data_path, index=False)
feature_summary.to_csv(feature_summary_path)
summary_stats.to_frame("value").to_csv(summary_stats_path)
acf_table.to_csv(acf_path, index=False)
Path(split_path).write_text(json.dumps(split_metadata, indent=2, default=str))
model_fit["history"].to_csv(training_history_path, index=False)
test_forecasts.to_csv(test_forecasts_path, index=False)
metrics.to_csv(metrics_path, index=False)
residual_summary.to_frame("value").to_csv(residual_summary_path)
bucket_error.to_csv(bucket_error_path, index=False)
placebo_metrics.to_csv(placebo_metrics_path, index=False)
uncertainty_df.to_csv(uncertainty_path, index=False)
uncertainty_summary.to_frame("value").to_csv(uncertainty_summary_path)
regime_diagnostics.to_csv(regime_path, index=False)
strategy_df.to_csv(strategy_path, index=False)
strategy_summary.to_csv(strategy_summary_path, index=False)
model_card.to_frame("value").to_csv(model_card_path)

manifest = {
    "dataset_name": "lstm_forecaster_outputs",
    "schema_version": "lstm_forecaster_v1",
    "created_at": pd.Timestamp.now(tz="UTC").isoformat(),
    "config": asdict(config),
    "model_card": model_card.to_dict(),
    "best_forecast_by_mse": metrics.sort_values("mse").iloc[0].to_dict(),
    "lstm_metrics": metrics[metrics["forecast"] == "forecast_lstm"].iloc[0].to_dict(),
    "placebo_summary": placebo_metrics.to_dict(orient="records"),
    "uncertainty_summary": uncertainty_summary.to_dict(),
    "notes": [
        "Synthetic data contains weak predictability and high noise.",
        "Forecast target is next-period return, not raw price.",
        "Scaling is fit on training data only.",
        "TensorFlow LSTM is used if available; otherwise fallback model is explicitly labelled.",
        "Placebo target sanity check is included.",
        "Residual bootstrap intervals are included as simple uncertainty estimates.",
        "Toy strategy is diagnostic only and not an alpha claim.",
        "Notebook explicitly avoids overclaiming return predictability."
    ]
}

manifest_path.write_text(json.dumps(manifest, indent=2, default=str))

data_path, test_forecasts_path, metrics_path, model_card_path, manifest_path

## 24. Limitations

### 24.1 Synthetic data

The notebook uses synthetic data.

Real market data adds:

- survivorship bias;
- corporate actions;
- contract rolls;
- missing timestamps;
- microstructure noise;
- transaction-cost uncertainty;
- structural breaks.

### 24.2 Weak target

Next-return prediction is extremely noisy.

Even statistically significant forecasts may be economically useless.

### 24.3 LSTM availability

A true LSTM is trained only when TensorFlow is available.

Otherwise the fallback model is not an LSTM and should not be described as one.

### 24.4 Limited hyperparameter search

The architecture is intentionally compact.

A full research process would tune sequence length, features, units, dropout, learning rate, and retraining frequency.

### 24.5 No live walk-forward retraining

The model is trained once and tested once.

Production use requires walk-forward retraining and model governance.

### 24.6 Uncertainty estimate is simple

Residual bootstrap intervals are approximate.

A full probabilistic model would forecast conditional distributions.

### 24.7 Toy strategy is not execution-realistic

The cost model is deliberately simple.

Real trading requires spread, slippage, latency, market impact, liquidity, and capacity modelling.

## 25. Research frontier and extensions

Important extensions include:

1. **Walk-forward LSTM retraining**  
   Refit or update the neural model through time.

2. **Temporal convolutional networks**  
   Often more stable than recurrent models.

3. **Transformer encoders**  
   Useful for long context but easy to overfit.

4. **Probabilistic forecasting**  
   Predict return distributions or quantiles, not only mean returns.

5. **Multi-asset sequence models**  
   Use cross-asset returns, volatility, rates, FX, and commodities.

6. **Regime-conditioned neural networks**  
   Add HMM/GMM-HMM regime probabilities as features.

7. **Cost-aware loss functions**  
   Optimise losses closer to trading utility.

8. **Purged walk-forward validation**  
   Especially important when labels overlap.

9. **Model interpretability**  
   Feature attribution, permutation importance, and sensitivity diagnostics.

10. **Chinese futures sequence forecasting**  
   Include night sessions, rolls, price limits, product calendars, and L1 regime features.

## 26. Suggested follow-up notebooks

This notebook naturally leads to:

1. **03_08_tree_models_for_return_prediction**  
   Compare LSTM against random forest and gradient boosting.

2. **03_09_information_coefficient_analysis**  
   Evaluate forecast as a signal cross-sectionally and through time.

3. **03_10_walk_forward_model_validation**  
   Proper live-like retraining protocol.

4. **03_11_feature_importance_and_shap_for_alpha**  
   Interpret model features.

5. **05_01_vectorized_backtest_engine**  
   Test forecast-driven strategies with realistic costs.

6. **06_12_gpu_sequence_model_training**  
   Scale sequence models efficiently.

7. **07_01_multi_asset_cta_research_pipeline**  
   Use sequence forecasts as one component of a broader alpha/risk system.

## 27. Summary

This notebook built an LSTM-style return forecaster while avoiding overclaiming predictability.

It showed how to:

1. simulate weakly predictable financial returns;
2. create leakage-safe sequence features;
3. split data chronologically;
4. scale features using training data only;
5. build sequence windows;
6. train an LSTM if TensorFlow is available;
7. use a clearly labelled fallback if not;
8. compare against simple baselines;
9. evaluate MSE, MAE, information coefficient, and directional accuracy;
10. run a placebo-label sanity check;
11. estimate simple residual-bootstrap forecast intervals;
12. diagnose performance by volatility regime;
13. test transaction-cost sensitivity;
14. document the model with an anti-overclaim model card.

The key computational takeaway:

> A sequence model is only as credible as its validation protocol.

The key financial takeaway:

> Return predictability should be treated as fragile, noisy, and regime-dependent until proven otherwise after baselines and costs.

## 28. Further reading

- Hochreiter and Schmidhuber, "Long Short-Term Memory."
- Hyndman and Athanasopoulos, *Forecasting: Principles and Practice.*
- Hamilton, *Time Series Analysis.*
- Literature on walk-forward validation, financial machine learning leakage, probabilistic forecasting, and neural sequence models.